# Contextual vs isolated detail embeddings, and what they change about compression

`notebooks/gpt_oss_detail_centrality_retention.ipynb` asks whether the details a summary keeps
are the *central* ones, where centrality is measured with GPT-OSS-20B contextual vectors. It cannot
run on this machine: it needs `original_detail_representation_coverage.csv.gz` and the
`contextual_detail_features_*.npz` / `isolated_detail_features_*.npz` shards, which only exist on
the HPC (`hpc/run_gpt_oss_detail_centrality.slurm`). `mechanistic_interpretability/outputs/` is
empty locally apart from `.gitkeep`.

So this notebook answers the same *question* with data that is available locally, and sets up the
comparison so the GPT-OSS numbers can be dropped in later.

### The question

`scripts/analyze_gpt_oss_contextualization_age.py` defines the contrast:

```python
details["context_isolated_cosine"] = np.einsum("ij,ij->i", context_vectors[ci], isolated_vectors[ii])
details["contextualization_shift"] = 1.0 - details["context_isolated_cosine"]
```

A detail has two vectors. The **isolated** vector is what the detail means on its own. The
**contextual** vector is what it means once the model has read the story up to that point. The gap
between them — the contextualization shift — is how much the surrounding story changes the detail.

That contrast matters for the compression work because the detail-compression analysis in
`notebooks/detail_compression_preservation.ipynb` is built entirely on TF-IDF cosine, which is a
*purely isolated, purely lexical* representation. A bag of words has no notion of context at all.
If the conclusion "summaries keep the central details" only holds in that space, it is a fact about
TF-IDF rather than about summarisation.

### What this notebook does

It holds the stories, the summaries and the matching rule fixed and varies only the vector space:

| space | what it is | context? |
|---|---|---|
| `tfidf` | bag of unigrams + bigrams, fit per story | none |
| `isolated` | MiniLM, detail embedded alone | none |
| `contextual` | MiniLM, detail embedded after its story prefix | causal prefix |

`scripts/extract_contextual_isolated_details.py` builds the two dense spaces. A detail is always
pooled over *its own tokens*; the only difference is whether the preceding story text is prepended
to the input. That reproduces GPT-OSS's causal prefix conditioning with an encoder small enough to
run on CPU offline.

Matching is **top-1 in all three spaces**: an original detail counts as retained when at least one
summary detail picks it as nearest neighbour. Top-1 is the rule the GPT-OSS notebook uses, and it
needs no similarity threshold — which is what makes three spaces with very different cosine scales
comparable at all.

### What this is not

MiniLM is a 6-layer bidirectional encoder; GPT-OSS-20B is a large causal decoder. The *direction*
and *robustness* of effects should transfer; the magnitudes should not be read as estimates of the
GPT-OSS numbers. The HPC run remains the thing that answers the question for GPT-OSS itself.

In [ ]:
from __future__ import annotations

import os
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats

warnings.filterwarnings('ignore')

# Where the extracted MiniLM tables live (pipeline_output/ is not in git) and where outputs go.
DATA_ROOT = Path(os.environ.get('STORY_FEATURES_DATA_ROOT', '/Users/tzl/repos/story-features'))
RESULTS = Path(os.environ.get('RESULTS_DIR', 'results')).resolve()
OUT_DIR = RESULTS / 'analysis/representation/detail_compression/minilm_proxy'

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda v: f'{v:,.4f}')

COMBINED_DIR = DATA_ROOT / os.environ.get('PM3_COMBINED_DIR', 'pipeline_output/pm3_grounding_combined')
CONTRAST_DIR = COMBINED_DIR / 'representation_contrast'
FIG_DIR = OUT_DIR / 'figures'
TABLE_DIR = OUT_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

REPS = ['tfidf', 'isolated', 'contextual']
REP_LABEL = {'tfidf': 'TF-IDF (lexical, no context)',
             'isolated': 'MiniLM isolated',
             'contextual': 'MiniLM contextual'}
REP_PALETTE = dict(zip(REPS, ['#8C8C8C', '#4C72B0', '#C44E52']))

def save_fig(name):
    plt.savefig(FIG_DIR / f'{name}.png', dpi=150, bbox_inches='tight')

def save_table(frame, name):
    frame.to_csv(TABLE_DIR / f'{name}.csv', index=False)
    return frame

config = json.loads((CONTRAST_DIR / 'extraction_config.json').read_text())
details = pd.read_csv(CONTRAST_DIR / 'detail_retention_by_representation.csv.gz', low_memory=False)
stories = pd.read_csv(CONTRAST_DIR / 'story_retention_by_representation.csv', low_memory=False)
meta = pd.read_csv(CONTRAST_DIR / 'detail_metadata.csv.gz', low_memory=False)

selection = stories[stories['representation'].isin(REPS)].copy()
agreement = stories[stories['representation'].str.startswith('agreement::')].copy()

print('extraction config:')
for key, value in config.items():
    print(f'  {key:20s} {value}')
print()
print(f'stories analysed      : {details["story_key"].nunique():,}')
print(f'original details      : {details["story_key"].nunique() and len(details) // 3:,}')
print(f'source-age range      : {details["source_age"].min():.1f} - {details["source_age"].max():.1f}')

## 1. What the contextualization shift actually measures

Before using the shift as a variable, it is worth seeing how it behaves. Two things must be true
for it to be meaningful: it should not be a constant, and it should not simply be a restatement of
"how far into the story this detail is".

Note one artefact by construction: the **first detail of every record has shift exactly 0**, because
it has no prefix to be conditioned on. Every model below is checked with those rows dropped.

In [ ]:
orig_meta = meta[meta['record_type'].eq('original')].copy()
summ_meta = meta[meta['record_type'].eq('summary')].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

for frame, label, colour in [(orig_meta, 'Original details', '#4C72B0'),
                             (summ_meta, 'Summary details', '#C44E52')]:
    axes[0].hist(frame['contextualization_shift'], bins=60, alpha=0.55,
                 density=True, label=f'{label} (n={len(frame):,})', color=colour)
axes[0].set_xlabel('contextualization shift  =  1 - cos(contextual, isolated)')
axes[0].set_ylabel('density')
axes[0].set_title('How much context moves a detail')
axes[0].legend(fontsize=8)

binned = orig_meta[orig_meta['detail_index'] > 0].copy()
binned['pos_bin'] = pd.cut(binned['position_norm'], np.linspace(0, 1, 11))
by_pos = binned.groupby('pos_bin', observed=True)['contextualization_shift'].agg(['mean', 'sem'])
axes[1].errorbar(range(len(by_pos)), by_pos['mean'], yerr=1.96 * by_pos['sem'],
                 marker='o', color='#4C72B0')
axes[1].set_xlabel('position in story (decile)')
axes[1].set_ylabel('mean shift')
axes[1].set_title('Shift grows as the prefix grows')

word_bins = orig_meta[orig_meta['detail_index'] > 0].copy()
word_bins['word_bin'] = pd.qcut(word_bins['detail_words'], 8, duplicates='drop')
by_words = word_bins.groupby('word_bin', observed=True)['contextualization_shift'].agg(['mean', 'sem'])
axes[2].errorbar(range(len(by_words)), by_words['mean'], yerr=1.96 * by_words['sem'],
                 marker='o', color='#55A868')
axes[2].set_xticks(range(len(by_words)))
axes[2].set_xticklabels([f'{int(iv.right)}' for iv in by_words.index], fontsize=8)
axes[2].set_xlabel('detail length (words, upper edge of octile)')
axes[2].set_ylabel('mean shift')
axes[2].set_title('Short details are moved more by context')

plt.tight_layout()
save_fig('ctx_shift_behaviour')
plt.show()

nonfirst = orig_meta[orig_meta['detail_index'] > 0]
print(f'original details, excluding first-in-story (n={len(nonfirst):,})')
print(f'  mean shift            {nonfirst["contextualization_shift"].mean():.4f}')
print(f'  sd                    {nonfirst["contextualization_shift"].std():.4f}')
print(f'  spearman vs position  {stats.spearmanr(nonfirst["position_norm"], nonfirst["contextualization_shift"]).statistic:+.3f}')
print(f'  spearman vs words     {stats.spearmanr(nonfirst["detail_words"], nonfirst["contextualization_shift"]).statistic:+.3f}')

## 2. Do the three spaces agree about which details are central?

Centrality is the mean cosine of a detail to the other details in the same story — the quantity the
compression notebook and the GPT-OSS notebook both build on. If the three spaces rank details
similarly, the choice of representation is cosmetic. If they do not, then "central" is not a single
well-defined property of a detail.

Spearman correlation is computed **within each story** and then summarised across stories, because
centrality is only ever compared within a story.

In [ ]:
wide = details.pivot_table(index=['story_key', 'orig_idx'], columns='representation', values='centrality')

rows = []
for left, right in [('tfidf', 'isolated'), ('tfidf', 'contextual'), ('isolated', 'contextual')]:
    values = []
    for _, group in wide.groupby(level=0):
        if len(group) > 3:
            rho = stats.spearmanr(group[left], group[right]).statistic
            if np.isfinite(rho):
                values.append(rho)
    values = np.array(values)
    rows.append({'pair': f'{left} vs {right}', 'n_stories': len(values),
                 'median_rho': np.median(values), 'mean_rho': values.mean(),
                 'pct_rho_below_0.5': float((values < 0.5).mean())})
centrality_agreement = save_table(pd.DataFrame(rows), 'ctx_centrality_rank_agreement')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

for (left, right), colour in zip([('tfidf', 'isolated'), ('tfidf', 'contextual'), ('isolated', 'contextual')],
                                 ['#8C8C8C', '#C44E52', '#4C72B0']):
    values = []
    for _, group in wide.groupby(level=0):
        if len(group) > 3:
            rho = stats.spearmanr(group[left], group[right]).statistic
            if np.isfinite(rho):
                values.append(rho)
    axes[0].hist(values, bins=40, histtype='step', linewidth=2,
                 label=f'{left} vs {right}', color=colour, density=True)
axes[0].axvline(0, color='k', linewidth=0.8)
axes[0].set_xlabel('within-story Spearman rho of detail centrality')
axes[0].set_ylabel('density')
axes[0].set_title('Agreement on who is central')
axes[0].legend(fontsize=8)

scales = details.groupby('representation')['centrality'].agg(['mean', 'std'])
axes[1].bar([REP_LABEL[r] for r in REPS], [scales.loc[r, 'mean'] for r in REPS],
            yerr=[scales.loc[r, 'std'] for r in REPS], color=[REP_PALETTE[r] for r in REPS])
axes[1].set_ylabel('mean detail centrality (raw cosine)')
axes[1].set_title('Raw centrality is on wildly different scales')
axes[1].tick_params(axis='x', labelsize=8)

plt.tight_layout()
save_fig('ctx_centrality_agreement')
plt.show()

display(centrality_agreement)
print('Raw cosine scales differ by more than an order of magnitude, which is why every')
print('comparison below uses within-story percentile ranks rather than raw cosine.')

## 3. Do the three spaces agree about which details survive?

Same stories, same summaries, same top-1 rule — only the vector space changes. Cohen's kappa is
computed per story against chance agreement implied by each space's own retention rate, so a high
kappa means genuine agreement rather than two spaces both keeping roughly 20% of details.

In [ ]:
rate_table = save_table(
    selection.groupby('representation')
    .agg(n_stories=('story_key', 'nunique'),
         retention_rate=('retention_rate', 'mean'),
         mean_best_cosine=('mean_best_cosine', 'mean'))
    .reindex(REPS).reset_index(),
    'ctx_retention_rates')

agree_table = save_table(
    agreement.groupby('representation')[['jaccard', 'observed_agreement', 'cohens_kappa']]
    .mean().reset_index(),
    'ctx_retention_agreement')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

axes[0].bar([REP_LABEL[r] for r in REPS],
            rate_table.set_index('representation').loc[REPS, 'retention_rate'],
            color=[REP_PALETTE[r] for r in REPS])
axes[0].set_ylabel('share of original details retained')
axes[0].set_title('All three spaces keep a similar fraction')
axes[0].tick_params(axis='x', labelsize=8)

pairs = agree_table['representation'].str.replace('agreement::', '', regex=False)
x = np.arange(len(pairs))
axes[1].bar(x - 0.2, agree_table['jaccard'], width=0.4, label='Jaccard', color='#4C72B0')
axes[1].bar(x + 0.2, agree_table['cohens_kappa'], width=0.4, label="Cohen's kappa", color='#C44E52')
axes[1].set_xticks(x)
axes[1].set_xticklabels(pairs, fontsize=8)
axes[1].set_ylabel('mean across stories')
axes[1].set_ylim(0, 1)
axes[1].set_title('...but not the same fraction')
axes[1].legend(fontsize=8)

plt.tight_layout()
save_fig('ctx_retention_agreement')
plt.show()

display(rate_table)
display(agree_table)

## 4. Does the compression conclusion survive the representation swap?

This is the load-bearing test. The compression notebook's headline is that retained details are
more central than omitted ones. Here that same contrast is computed in each space, using
**within-story centrality percentile** so the three are on a common 0–1 scale, and tested with a
paired Wilcoxon across stories with Holm correction.

If the effect is positive and significant in all three, the conclusion is a property of
summarisation. If it appears only in some, it is a property of the representation.

In [ ]:
gaps = (details.pivot_table(index=['story_key', 'representation'], columns='retained',
                            values='centrality_pct', aggfunc='mean')
        .rename(columns={True: 'retained', False: 'omitted'}).dropna().reset_index())
gaps['gap'] = gaps['retained'] - gaps['omitted']

rows = []
for rep in REPS:
    group = gaps[gaps['representation'].eq(rep)]['gap']
    result = stats.wilcoxon(group)
    rows.append({'representation': rep, 'n_stories': len(group),
                 'mean_gap': group.mean(), 'median_gap': group.median(),
                 'cohens_dz': group.mean() / group.std(),
                 'ci_lo': group.mean() - 1.96 * group.sem(),
                 'ci_hi': group.mean() + 1.96 * group.sem(),
                 'p_raw': result.pvalue})
gap_table = pd.DataFrame(rows)
order = np.argsort(gap_table['p_raw'].to_numpy())
holm = np.empty(len(gap_table))
running = 0.0
for rank, idx in enumerate(order):
    running = max(running, (len(gap_table) - rank) * gap_table['p_raw'].iloc[idx])
    holm[idx] = min(running, 1.0)
gap_table['p_holm'] = holm
gap_table['significant'] = gap_table['p_holm'] < 0.05
gap_table = save_table(gap_table, 'ctx_centrality_gap_by_representation')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

y = np.arange(len(REPS))
sub = gap_table.set_index('representation').loc[REPS]
axes[0].errorbar(sub['mean_gap'], y,
                 xerr=[sub['mean_gap'] - sub['ci_lo'], sub['ci_hi'] - sub['mean_gap']],
                 fmt='o', capsize=4, color='#333333')
for i, rep in enumerate(REPS):
    axes[0].scatter(sub.loc[rep, 'mean_gap'], i, s=90, color=REP_PALETTE[rep], zorder=3)
axes[0].axvline(0, color='k', linewidth=1, linestyle='--')
axes[0].set_yticks(y)
axes[0].set_yticklabels([REP_LABEL[r] for r in REPS], fontsize=9)
axes[0].set_xlabel('centrality percentile gap (retained - omitted), 95% CI')
axes[0].set_title('Retained details are more central in every space')

for rep in REPS:
    axes[1].hist(gaps[gaps['representation'].eq(rep)]['gap'], bins=40, histtype='step',
                 linewidth=2, density=True, color=REP_PALETTE[rep], label=REP_LABEL[rep])
axes[1].axvline(0, color='k', linewidth=1, linestyle='--')
axes[1].set_xlabel('per-story centrality percentile gap')
axes[1].set_ylabel('density')
axes[1].set_title('Per-story distribution')
axes[1].legend(fontsize=8)

plt.tight_layout()
save_fig('ctx_centrality_gap_forest')
plt.show()

display(gap_table[['representation', 'n_stories', 'mean_gap', 'cohens_dz', 'p_holm', 'significant']])

## 5. Contextualization shift as its own predictor of survival

The previous sections treat the contextual vectors as an alternative way to measure centrality. But
having both vectors gives something the compression analysis never had: a per-detail measure of
**how context-dependent a detail is**, independent of how central it is.

The question this answers: does a summary prefer details that mean the same thing on their own as
they do in the story? A logistic model on retention, clustered by story, with position, length and
centrality as controls.

In [ ]:
model_frame = details[details['representation'].eq('contextual')].copy()
model_frame['y'] = model_frame['retained'].astype(int)
model_frame['log_words'] = np.log1p(model_frame['detail_words'])
model_frame['shift_z'] = ((model_frame['contextualization_shift'] - model_frame['contextualization_shift'].mean())
                          / model_frame['contextualization_shift'].std())
model_frame['centrality_z'] = (model_frame['centrality_pct'] - 0.5) / model_frame['centrality_pct'].std()

specs = {
    'shift only': 'y ~ shift_z',
    '+ position, length': 'y ~ shift_z + position_norm + log_words',
    '+ centrality': 'y ~ shift_z + centrality_z + position_norm + log_words',
}
rows = []
fitted = {}
for name, formula in specs.items():
    groups = model_frame['story_key'].astype('category').cat.codes
    model = smf.logit(formula, data=model_frame).fit(disp=False, cov_type='cluster',
                                                     cov_kwds={'groups': groups})
    fitted[name] = model
    for term in model.params.index:
        if term == 'Intercept':
            continue
        rows.append({'spec': name, 'term': term,
                     'odds_ratio': np.exp(model.params[term]),
                     'ci_lo': np.exp(model.conf_int().loc[term, 0]),
                     'ci_hi': np.exp(model.conf_int().loc[term, 1]),
                     'z': model.tvalues[term], 'p': model.pvalues[term]})

# The first detail of each story has shift 0 by construction; confirm it is not driving anything.
no_first = model_frame[model_frame['orig_idx'] > 0]
robust = smf.logit('y ~ shift_z + centrality_z + position_norm + log_words', data=no_first).fit(
    disp=False, cov_type='cluster', cov_kwds={'groups': no_first['story_key'].astype('category').cat.codes})
for term in robust.params.index:
    if term == 'Intercept':
        continue
    rows.append({'spec': 'excl. first detail', 'term': term,
                 'odds_ratio': np.exp(robust.params[term]),
                 'ci_lo': np.exp(robust.conf_int().loc[term, 0]),
                 'ci_hi': np.exp(robust.conf_int().loc[term, 1]),
                 'z': robust.tvalues[term], 'p': robust.pvalues[term]})

survival_table = save_table(pd.DataFrame(rows), 'ctx_survival_logit')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

full = survival_table[survival_table['spec'].eq('+ centrality')].set_index('term')
terms = ['shift_z', 'centrality_z', 'position_norm', 'log_words']
y = np.arange(len(terms))
axes[0].errorbar(full.loc[terms, 'odds_ratio'], y,
                 xerr=[full.loc[terms, 'odds_ratio'] - full.loc[terms, 'ci_lo'],
                       full.loc[terms, 'ci_hi'] - full.loc[terms, 'odds_ratio']],
                 fmt='o', capsize=4, color='#333333')
axes[0].axvline(1, color='k', linestyle='--', linewidth=1)
axes[0].set_yticks(y)
axes[0].set_yticklabels(['contextualization shift (+1 SD)', 'centrality (+1 SD)',
                         'position in story (0 to 1)', 'log detail words'], fontsize=9)
axes[0].set_xscale('log')
axes[0].set_xlabel('odds ratio for being retained (log scale)')
axes[0].set_title('Shift and centrality pull in opposite directions')

deciles = no_first.copy()
deciles['shift_bin'] = pd.qcut(deciles['contextualization_shift'], 10, labels=False, duplicates='drop')
by_bin = deciles.groupby('shift_bin').agg(retention=('y', 'mean'), sem=('y', 'sem'),
                                          shift=('contextualization_shift', 'mean'))
axes[1].errorbar(by_bin['shift'], by_bin['retention'], yerr=1.96 * by_bin['sem'],
                 marker='o', color='#C44E52')
axes[1].set_xlabel('contextualization shift (decile mean)')
axes[1].set_ylabel('share retained')
axes[1].set_title('Context-dependent details are dropped')

plt.tight_layout()
save_fig('ctx_survival_logit')
plt.show()

display(survival_table[survival_table['term'].isin(['shift_z', 'centrality_z'])]
        [['spec', 'term', 'odds_ratio', 'ci_lo', 'ci_hi', 'p']])
print()
print(f"spearman(shift, centrality percentile) = "
      f"{stats.spearmanr(model_frame['contextualization_shift'], model_frame['centrality_pct']).statistic:+.3f}")
print('Near zero: the shift is close to orthogonal to centrality, so it is a genuinely')
print('new axis rather than a re-description of the centrality result.')

### 5.1 Retention and context-dependence across source age

**How the shift is calculated.** Every detail is encoded twice with the same MiniLM model (all-MiniLM-L6-v2):

1. **isolated**: the input is `[CLS] detail [SEP]`.
2. **contextual**: the input is `[CLS] preceding story text + detail [SEP]`, where the preceding text is up to the last 180 words of the details before it (further cut to fit 256 tokens).

In both cases only the detail's own tokens are averaged (final layer, mean-pooled), and the vector is length-normalised. The shift is `1 - cosine(contextual, isolated)`: 0 means the surrounding text did not change how the model represents the detail, larger means it did. The first detail of a story has no preceding text, so its two inputs are identical and its shift is exactly 0 (those rows are dropped). MiniLM reads in both directions, but only the preceding text is supplied, so this imitates a causal reader's prefix. It is not the GPT-OSS measure, where the same detail is read inside the whole text at a given layer and compared with the detail encoded alone at that layer.

**What is plotted.** Age is a continuous axis: one value per story, smoothed with a Gaussian kernel (width 0.75 years) and a story-resampling 95% band. Three measures, all comparing details *within* a story, so book length and book age cannot leak in:

- **AUC**: the chance that a randomly chosen *dropped* detail has a larger shift than a randomly chosen *kept* one. 0.5 means no relation; above 0.5 means context-dependent details are dropped.
- **Retention ratio (log2)**: retention among a story's low-shift half over retention among its high-shift half. 0 means no relation; positive means low-shift details are kept more.
- **Odds ratio**: for being kept, per +1 SD of shift, with centrality, position and detail length held fixed, estimated with a kernel window that slides along age.

Shift rises with position in the story and varies with detail length, so the first two are shown for the raw shift and for the shift **adjusted** for position and length (grey and red).

In [ ]:
import sys
for _here in [Path(os.environ.get('STORY_FEATURES_ROOT', '.')).resolve(), Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_here / 'src/shared/retention_shift.py').exists():
        sys.path.insert(0, str(_here / 'src'))
        break
from shared.retention_shift import adjust_shift, smooth_with_band, story_measures, windowed_odds

work = no_first.copy()
work['shift'] = work['contextualization_shift']
work['detail_len'] = work['detail_words']
work['log_len'] = np.log1p(work['detail_words'])
work['shift_adj'] = adjust_shift(work)

raw_story = story_measures(work, shift='shift')
adj_story = story_measures(work, shift='shift_adj')
story_ages = work.drop_duplicates('story_key')['source_age']
lo_age, hi_age = np.quantile(story_ages, [0.03, 0.97])
GRID = np.round(np.arange(np.ceil(lo_age * 2) / 2, hi_age + 1e-9, 0.25), 2)
BANDWIDTH = 0.75
print(f'{work.story_key.nunique()} stories, {len(raw_story)} with at least 2 kept and 2 dropped details; age grid {GRID.min():g} to {GRID.max():g}')
display(adj_story.describe().loc[['count', 'mean', '50%'], ['auc', 'log2_rr', 'retained', 'omitted']].round(3))

In [ ]:
auc_raw = smooth_with_band(raw_story, 'auc', GRID, BANDWIDTH, seed=1)
auc_adj = smooth_with_band(adj_story, 'auc', GRID, BANDWIDTH, seed=2)
rr_raw = smooth_with_band(raw_story, 'log2_rr', GRID, BANDWIDTH, seed=3)
rr_adj = smooth_with_band(adj_story, 'log2_rr', GRID, BANDWIDTH, seed=4)
odds = windowed_odds(work, GRID[::2], BANDWIDTH, n_boot=100, seed=5, shift='shift_adj')
save_table(pd.concat([auc_raw.assign(measure='auc', shift='raw'), auc_adj.assign(measure='auc', shift='adjusted'),
                      rr_raw.assign(measure='log2_rr', shift='raw'), rr_adj.assign(measure='log2_rr', shift='adjusted'),
                      odds.assign(measure='odds_ratio', shift='adjusted')]), 'ctx_age_continuous_curves')

def draw(ax, curve, color, label):
    ax.fill_between(curve['age'], curve['lo'], curve['hi'], color=color, alpha=0.18, linewidth=0)
    ax.plot(curve['age'], curve['estimate'], color=color, linewidth=2.2, label=label)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, (raw_curve, adj_curve, table, column, null, ylabel, title) in zip(axes[:2], [
        (auc_raw, auc_adj, adj_story, 'auc', 0.5, 'AUC: dropped detail has the larger shift', 'Dropped details are more context-dependent'),
        (rr_raw, rr_adj, adj_story, 'log2_rr', 0.0, 'log2 retention: low-shift half / high-shift half', 'Low-shift details are kept more often')]):
    ax.scatter(table['source_age'], table[column], s=9, color='#C44E52', alpha=0.12, linewidths=0)
    draw(ax, raw_curve, '#8C8C8C', 'raw shift')
    draw(ax, adj_curve, '#C44E52', 'adjusted for position and length')
    ax.axhline(null, color='k', linestyle='--', linewidth=1)
    ax.set_xlabel('source age (years)'); ax.set_ylabel(ylabel); ax.set_title(title, fontsize=10)
axes[0].legend(fontsize=8, loc='upper left')
draw(axes[2], odds, '#C44E52', 'adjusted shift')
axes[2].axhline(1, color='k', linestyle='--', linewidth=1)
axes[2].set_yscale('log')
axes[2].set_xlabel('source age (years)'); axes[2].set_ylabel('odds ratio for being kept, per +1 SD of shift')
axes[2].set_title('Odds ratio, sliding along age', fontsize=10)
for ax in axes:
    ax.plot(story_ages, np.full(len(story_ages), ax.get_ylim()[0]), '|', color='#555555', alpha=0.25, markersize=8)
    ax.set_xlim(GRID.min(), GRID.max())
plt.tight_layout()
save_fig('ctx_survival_by_age_continuous')
plt.show()

In [ ]:
def age_slope(table, column, n_boot=1000, seed=0):
    # Weighted linear trend of a story measure on age (per year), with a story-resampling interval.
    x, y, w = (table[c].to_numpy(float) for c in ['source_age', column, 'weight'])
    fit = lambda i: np.polyfit(x[i], y[i], 1, w=np.sqrt(w[i]))[0]
    rng = np.random.default_rng(seed)
    boots = [fit(rng.integers(0, len(x), len(x))) for _ in range(n_boot)]
    return fit(np.arange(len(x))), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

rows = []
for label, table in [('raw shift', raw_story), ('adjusted shift', adj_story)]:
    for column, name in [('auc', 'AUC'), ('log2_rr', 'log2 retention ratio')]:
        slope, lo, hi = age_slope(table, column)
        rows.append({'shift': label, 'measure': name, 'stories': len(table), 'change per year of age': slope, 'ci_lo': lo, 'ci_hi': hi})
trend = save_table(pd.DataFrame(rows), 'ctx_age_trend_of_measures')
display(trend.round(4))
print('A change per year whose interval excludes 0 means the link between context-dependence and being dropped changes with age.')
print('The AUC and ratio compare details inside one story, so a longer book for older readers does not produce a trend by itself.')

Reading section 5.1:

- Flat curves at 0.5 (AUC), 0 (log ratio) or 1 (odds ratio) mean context-dependence does not predict being dropped. Curves on one side of that line that stay level across age mean it does, equally at every age.
- A slope in the trend table whose interval excludes 0 is a change with age. The band on the curves shows how sure the shape is at each age.
- Few stories are older than 6, so the right end is wide. The tick marks at the bottom show where the stories are.
- If the raw and adjusted curves differ, part of the raw link is position or detail length.
- The shift is mean-pooled MiniLM with and without a text prefix, a local stand-in for GPT-OSS. Read direction and age pattern, not magnitude.

## 6. Does any of this move with the age of the source book?

The GPT-OSS notebook's primary model is a level model of centrality with a story random intercept
and a secondary `centrality_gap ~ source_age_centered` slope. The age-trend notebook established
that raw age slopes in this corpus are largely a story-length artefact: books for older readers are
longer, and length drives compression.

Age is centred at 4 here rather than the GPT-OSS script's 8, because this corpus is concentrated at
ages 2–6. Every slope is reported with and without a `log(number of details)` control.

In [ ]:
story_level = (details[details['representation'].eq('contextual')]
               .groupby(['story_key', 'video_id', 'source_age'])
               .agg(mean_shift=('contextualization_shift', 'mean'),
                    retention=('retained', 'mean'),
                    n_details=('orig_idx', 'size'))
               .reset_index().dropna(subset=['source_age']))
story_level['age_c'] = story_level['source_age'] - 4.0
story_level['log_n'] = np.log(story_level['n_details'])

gap_wide = gaps[gaps['representation'].eq('contextual')].set_index('story_key')['gap']
story_level['centrality_gap'] = story_level['story_key'].map(gap_wide)

rows = []
clusters = story_level['video_id'].astype('category').cat.codes
for outcome in ['mean_shift', 'retention', 'centrality_gap']:
    for label, formula in [('raw', f'{outcome} ~ age_c'), ('length-controlled', f'{outcome} ~ age_c + log_n')]:
        frame = story_level.dropna(subset=[outcome])
        model = smf.ols(formula, data=frame).fit(
            cov_type='cluster', cov_kwds={'groups': frame['video_id'].astype('category').cat.codes})
        rows.append({'outcome': outcome, 'spec': label, 'n': len(frame),
                     'slope_per_year': model.params['age_c'],
                     'ci_lo': model.conf_int().loc['age_c', 0],
                     'ci_hi': model.conf_int().loc['age_c', 1],
                     't': model.tvalues['age_c'], 'p': model.pvalues['age_c']})
age_table = save_table(pd.DataFrame(rows), 'ctx_age_slopes')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, outcome, title in zip(axes, ['mean_shift', 'retention', 'centrality_gap'],
                              ['Mean contextualization shift', 'Retention rate',
                               'Centrality gap (retained - omitted)']):
    frame = story_level.dropna(subset=[outcome])
    binned = frame.groupby('source_age')[outcome].agg(['mean', 'sem', 'size'])
    binned = binned[binned['size'] >= 5]
    ax.errorbar(binned.index, binned['mean'], yerr=1.96 * binned['sem'], marker='o', color='#4C72B0')
    ax.set_xlabel('source age (years)')
    ax.set_title(title, fontsize=10)
    raw = age_table[(age_table['outcome'] == outcome) & (age_table['spec'] == 'raw')].iloc[0]
    adj = age_table[(age_table['outcome'] == outcome) & (age_table['spec'] == 'length-controlled')].iloc[0]
    ax.annotate(f"raw {raw['slope_per_year']:+.4f}/yr (p={raw['p']:.1e})\n"
                f"adj {adj['slope_per_year']:+.4f}/yr (p={adj['p']:.1e})",
                xy=(0.03, 0.04), xycoords='axes fraction', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
plt.tight_layout()
save_fig('ctx_age_slopes')
plt.show()

display(age_table)
print(f"correlation(source age, log number of details) = "
      f"{story_level[['age_c', 'log_n']].corr().iloc[0, 1]:+.3f}")

## 7. Is the age trend a translation of the whole centrality distribution, or a change of its shape?

Section 6 leaves an explanation hanging. Detail centrality declines with source age, and the verbal
account usually offered for it is that older stories may have more heterogeneous detail
representations, shifting the entire centrality distribution downward. That sentence bundles four
separable claims, and they are not equally well supported.

1. **Location.** Mean detail centrality falls as source age rises. That is the starting point rather
   than a finding; section 6 already establishes it.
2. **Pure shift.** The word *entire* asserts that every part of the distribution moves down by the
   same amount. The alternative is that the distribution changes shape, with the top of it falling
   faster than the bottom or the reverse. A translation and a stretch say different things about
   what happens to a story's details, so the two have to be told apart rather than assumed.
3. **Heterogeneity.** The proposed mechanism is that an older story's details are genuinely more
   spread out in embedding space. Mean cosine is a poor witness for that, because it is the very
   quantity being explained. The claim needs measures of spread that are not rescalings of the mean.
4. **Length.** Books for older readers are longer, and a longer story averages its centrality over
   more and more varied pairs. Mean pairwise cosine therefore falls with the number of details for a
   reason that has nothing to do with the reader. In this corpus the correlation between source age
   and log number of details is about +0.28, so "heterogeneous because longer" and "heterogeneous per
   detail at matched length" are badly entangled and have to be separated. That is the crux, and
   sub-section 7.6 is the test that decides it.

Everything below uses raw within-story cosine centrality rather than the percentile version, because
percentiles are uniform within a story by construction and so destroy exactly the distributional
information at issue. Both dense spaces are L2-normalised, so nothing that follows can be a
vector-norm effect; the overall elevation of cosines is set by the anisotropy of the encoder, which
is a fixed property of the model and layer and cannot itself vary with the age of a book.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from statsmodels.regression.quantile_regression import QuantReg

AGE_EDGES = [0.0, 3.25, 4.25, 5.25, 99.0]
AGE_BANDS = ['1.5-3.0', '3.5-4.0', '4.5-5.0', '5.5-11.5']
BAND_PALETTE = dict(zip(AGE_BANDS, sns.color_palette('rocket_r', len(AGE_BANDS))))
MATCHED_COLOUR = '#55A868'
MATCH_SIZE = 30
MATCH_DRAWS = 100
BOOT_DRAWS = 200
PERM_DRAWS = 500
SEED = 20260928

def mean_offdiag(similarity):
    return (similarity.sum(axis=1) - np.diag(similarity)) / (similarity.shape[0] - 1)

story_frame = (details[details['representation'].eq('contextual')]
               .groupby(['story_key', 'video_id', 'source_age'], as_index=False)
               .agg(n_details=('orig_idx', 'size')))
story_frame['age_c'] = story_frame['source_age'] - 4.0
story_frame['log_n'] = np.log(story_frame['n_details'])
story_frame['age_band'] = pd.cut(story_frame['source_age'], AGE_EDGES, labels=AGE_BANDS)
story_frame['cluster'] = story_frame['video_id'].astype('category').cat.codes

banded = details.merge(story_frame[['story_key', 'age_band', 'age_c', 'log_n', 'cluster']],
                       on='story_key', how='left')

originals = meta[meta['record_type'].eq('original')]
summary_frames = {key: frame for key, frame
                  in meta[meta['record_type'].eq('summary')].groupby('story_key', sort=False)}
with np.load(CONTRAST_DIR / 'detail_vectors.npz') as payload:
    dense = {name: payload[name].astype(np.float32) for name in ('isolated', 'contextual')}
for name in dense:
    dense[name] /= np.maximum(np.linalg.norm(dense[name], axis=1, keepdims=True), 1e-12)

# Every measure in this section is a function of a story's Gram matrix alone, so a submatrix is
# exactly the same story restricted to a subset of its details. Building the Gram matrices once is
# what makes the subsampling in 7.6 cheap.
gram_by_story = {}
for story_key, frame in originals.groupby('story_key', sort=False):
    frame = frame.sort_values('detail_index')
    original_texts = frame['detail_text'].astype(str).tolist()
    n_original = len(original_texts)
    summary_texts = summary_frames[story_key].sort_values('detail_index')['detail_text'].astype(str).tolist()
    matrix = TfidfVectorizer(lowercase=True, ngram_range=(1, 2)).fit_transform(original_texts + summary_texts)
    lexical = np.asarray(matrix[:n_original].todense(), dtype=np.float32)
    grams = {'tfidf': lexical @ lexical.T}
    for name in ('isolated', 'contextual'):
        vectors = dense[name][frame['vector_index'].to_numpy(int)]
        grams[name] = vectors @ vectors.T
    gram_by_story[story_key] = grams

reproduction_error = 0.0
for rep in REPS:
    stored = (details[details['representation'].eq(rep)]
              .sort_values(['story_key', 'orig_idx'])['centrality'].to_numpy())
    rebuilt = np.concatenate([mean_offdiag(gram_by_story[key][rep]) for key in sorted(gram_by_story)])
    reproduction_error = max(reproduction_error, float(np.abs(rebuilt - stored).max()))

display(story_frame.groupby('age_band', observed=True)
        .agg(stories=('story_key', 'size'), mean_age=('source_age', 'mean'),
             mean_details=('n_details', 'mean'), median_details=('n_details', 'median')).reset_index())
print(f'stories with Gram matrices             : {len(gram_by_story):,}')
print(f'max |recomputed - stored| centrality   : {reproduction_error:.2e}')
print(f'stories with at least {MATCH_SIZE} details       : '
      f'{int((story_frame["n_details"] >= MATCH_SIZE).sum())} of {len(story_frame)}')
print(f'corr(source age, log n details)        : '
      f'{story_frame[["source_age", "log_n"]].corr().iloc[0, 1]:+.3f}')

### 7.1 The distribution by source-age band

Source ages run from 1.5 to 11.5 but are heavily concentrated between 2 and 6, and age 8.0 has a
single story. Sparse cells are therefore pooled into four bands of roughly comparable size rather
than trusted individually, and every per-age curve later in the section requires at least five
stories in a cell.

The ridgelines below are the picture the hypothesis describes. What has to be established is whether
they are the same curve sliding left, or a curve that also changes shape as it slides.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, rep in zip(axes, REPS):
    subset = banded[banded['representation'].eq(rep)]
    grid = np.linspace(subset['centrality'].quantile(0.001), subset['centrality'].quantile(0.999), 400)
    for level, band in enumerate(AGE_BANDS):
        values = subset[subset['age_band'].eq(band)]['centrality'].to_numpy()
        density = stats.gaussian_kde(values)(grid)
        density = density / density.max()
        baseline = level * 0.9
        ax.fill_between(grid, baseline, baseline + density, color=BAND_PALETTE[band],
                        alpha=0.75, linewidth=0)
        ax.plot(grid, baseline + density, color='white', linewidth=1.0)
        ax.vlines(np.median(values), baseline, baseline + 0.95, color='white',
                  linewidth=1.4, linestyle='--')
    ax.set_yticks([level * 0.9 + 0.2 for level in range(len(AGE_BANDS))])
    ax.set_yticklabels([f'age {band}' for band in AGE_BANDS], fontsize=8)
    ax.set_xlabel('detail centrality (raw cosine)')
    ax.set_title(REP_LABEL[rep], fontsize=10)
axes[0].set_ylabel('source-age band')
plt.tight_layout()
save_fig('ctx_age_band_centrality_ridges')
plt.show()

band_summary = save_table(
    banded.groupby(['representation', 'age_band'], observed=True)['centrality']
    .agg(n_details='size', mean='mean', sd='std', q10=lambda s: s.quantile(0.10),
         median='median', q90=lambda s: s.quantile(0.90))
    .reset_index(), 'ctx_age_band_centrality_summary')
display(band_summary)

### 7.2 Quantile regression: does every part of the distribution move at the same rate?

This is the most direct test of the word *entire*. Detail centrality is regressed on centred source
age at each decile with `QuantReg`, separately per space. Under a pure downward translation all nine
slopes are equal. If the distribution is stretching or compressing, the slopes vary systematically
with the quantile level.

Standard errors come from a cluster bootstrap that resamples whole videos, because details are
nested inside stories and the iid errors `QuantReg` reports are several times too small. Two
summaries of non-constancy are given: the **tilt**, defined as the ninth-decile slope minus the
first-decile slope, and a **Wald test** of equality of all nine slopes taken against the bootstrap
covariance. Each specification is run with and without a `log(n details)` control, which turns out
to matter a great deal.

In [ ]:
QUANTILES = np.round(np.arange(0.1, 0.95, 0.1), 2)
rng = np.random.default_rng(SEED)
quantile_rows = []
constancy_rows = []
for rep in REPS:
    subset = banded[banded['representation'].eq(rep)]
    outcome = subset['centrality'].to_numpy()
    designs = {'raw': np.column_stack([np.ones(len(subset)), subset['age_c'].to_numpy()]),
               'length-controlled': np.column_stack([np.ones(len(subset)), subset['age_c'].to_numpy(),
                                                     subset['log_n'].to_numpy()])}
    codes = subset['cluster'].to_numpy()
    order = np.argsort(codes, kind='stable')
    bounds = np.searchsorted(codes[order], np.arange(codes.max() + 2))
    clusters = [block for block in (order[bounds[i]:bounds[i + 1]] for i in range(codes.max() + 1)) if len(block)]
    for spec, design in designs.items():
        observed = np.array([QuantReg(outcome, design).fit(q=q).params[1] for q in QUANTILES])
        iid_errors = np.array([QuantReg(outcome, design).fit(q=q).bse[1] for q in QUANTILES])
        draws = np.empty((BOOT_DRAWS, len(QUANTILES)))
        for draw in range(BOOT_DRAWS):
            index = np.concatenate([clusters[i] for i in rng.integers(0, len(clusters), len(clusters))])
            for position, q in enumerate(QUANTILES):
                draws[draw, position] = QuantReg(outcome[index], design[index]).fit(q=q).params[1]
        for position, q in enumerate(QUANTILES):
            quantile_rows.append({'representation': rep, 'spec': spec, 'quantile': q,
                                  'slope': observed[position],
                                  'ci_lo': np.quantile(draws[:, position], 0.025),
                                  'ci_hi': np.quantile(draws[:, position], 0.975),
                                  'cluster_boot_se': draws[:, position].std(ddof=1),
                                  'iid_se': iid_errors[position]})
        tilt_draws = draws[:, -1] - draws[:, 0]
        tilt = observed[-1] - observed[0]
        centred = tilt_draws - tilt_draws.mean()
        p_tilt = min(2 * min(float((centred >= abs(tilt)).mean()), float((centred <= -abs(tilt)).mean())), 1.0)
        contrast = np.eye(len(QUANTILES))[:-1] - np.eye(len(QUANTILES))[1:]
        deviation = contrast @ observed
        wald = float(deviation @ np.linalg.pinv(contrast @ np.cov(draws.T) @ contrast.T) @ deviation)
        constancy_rows.append({'representation': rep, 'spec': spec,
                               'slope_q10': observed[0], 'slope_q50': observed[len(QUANTILES) // 2],
                               'slope_q90': observed[-1], 'slope_spread': observed.max() - observed.min(),
                               'tilt_q90_minus_q10': tilt,
                               'tilt_ci_lo': np.quantile(tilt_draws, 0.025),
                               'tilt_ci_hi': np.quantile(tilt_draws, 0.975), 'p_tilt': p_tilt,
                               'wald_constant_slope': wald, 'wald_df': len(QUANTILES) - 1,
                               'p_wald': float(stats.chi2.sf(wald, len(QUANTILES) - 1)),
                               'median_cluster_boot_se': float(np.median(draws.std(axis=0, ddof=1))),
                               'median_iid_se': float(np.median(iid_errors))})
quantile_slopes = save_table(pd.DataFrame(quantile_rows), 'ctx_age_quantile_slopes')
quantile_constancy = save_table(pd.DataFrame(constancy_rows), 'ctx_age_quantile_constancy')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
for ax, rep in zip(axes, REPS):
    for spec, style, colour, alpha in [('raw', '-', REP_PALETTE[rep], 0.22),
                                       ('length-controlled', '--', MATCHED_COLOUR, 0.16)]:
        curve = quantile_slopes[quantile_slopes['representation'].eq(rep) & quantile_slopes['spec'].eq(spec)]
        ax.plot(curve['quantile'], curve['slope'], style, marker='o', markersize=4,
                color=colour, label=spec)
        ax.fill_between(curve['quantile'], curve['ci_lo'], curve['ci_hi'],
                        color=colour, alpha=alpha, linewidth=0)
    row = quantile_constancy[quantile_constancy['representation'].eq(rep)
                             & quantile_constancy['spec'].eq('raw')].iloc[0]
    ax.axhline(0, color='k', linewidth=0.8)
    ax.annotate(f"raw: tilt {row['tilt_q90_minus_q10']:+.4f} (p={row['p_tilt']:.3f})\n"
                f"raw: constant-slope Wald p={row['p_wald']:.1e}\n"
                f"adjusted: Wald p="
                f"{quantile_constancy[quantile_constancy['representation'].eq(rep) & quantile_constancy['spec'].eq('length-controlled')]['p_wald'].iloc[0]:.2f}",
                xy=(0.03, 0.05), xycoords='axes fraction', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_xlabel('quantile of detail centrality')
    ax.set_title(REP_LABEL[rep], fontsize=10)
    ax.legend(fontsize=8, loc='upper right')
axes[0].set_ylabel('slope of centrality on source age (per year)')
plt.tight_layout()
save_fig('ctx_age_quantile_slopes')
plt.show()

display(quantile_constancy[['representation', 'spec', 'slope_q10', 'slope_q50', 'slope_q90',
                           'tilt_q90_minus_q10', 'p_tilt', 'p_wald']])
print(f"cluster bootstrap standard errors are "
      f"{quantile_constancy['median_cluster_boot_se'].sum() / quantile_constancy['median_iid_se'].sum():.1f}x "
      f"the iid errors QuantReg reports, so the iid version would badly overstate precision.")

### 7.3 The shift function between the youngest and the oldest band

Doksum's shift function plots the difference between matched quantiles of two groups against the
quantile level. A pure location shift is a flat horizontal line at the size of the shift; any slope
in it is a change of shape. Confidence bands come from a story-level bootstrap of both bands, and
the tilt is the shift at the 0.95 quantile minus the shift at the 0.05 quantile.

In [ ]:
SHIFT_QUANTILES = np.round(np.arange(0.05, 0.96, 0.05), 2)
rng = np.random.default_rng(SEED + 1)
shift_rows = []
for rep in REPS:
    subset = banded[banded['representation'].eq(rep)]
    young = [frame['centrality'].to_numpy() for _, frame
             in subset[subset['age_band'].eq(AGE_BANDS[0])].groupby('story_key', sort=False)]
    old = [frame['centrality'].to_numpy() for _, frame
           in subset[subset['age_band'].eq(AGE_BANDS[-1])].groupby('story_key', sort=False)]
    observed = (np.quantile(np.concatenate(old), SHIFT_QUANTILES)
                - np.quantile(np.concatenate(young), SHIFT_QUANTILES))
    draws = np.empty((PERM_DRAWS, len(SHIFT_QUANTILES)))
    for draw in range(PERM_DRAWS):
        left = np.concatenate([young[i] for i in rng.integers(0, len(young), len(young))])
        right = np.concatenate([old[i] for i in rng.integers(0, len(old), len(old))])
        draws[draw] = np.quantile(right, SHIFT_QUANTILES) - np.quantile(left, SHIFT_QUANTILES)
    tilt_draws = draws[:, -1] - draws[:, 0]
    tilt = observed[-1] - observed[0]
    centred = tilt_draws - tilt_draws.mean()
    p_tilt = min(2 * min(float((centred >= abs(tilt)).mean()), float((centred <= -abs(tilt)).mean())), 1.0)
    for position, q in enumerate(SHIFT_QUANTILES):
        shift_rows.append({'representation': rep, 'quantile': q, 'delta': observed[position],
                           'ci_lo': np.quantile(draws[:, position], 0.025),
                           'ci_hi': np.quantile(draws[:, position], 0.975),
                           'tilt': tilt, 'p_tilt': p_tilt,
                           'n_young': int(sum(len(v) for v in young)),
                           'n_old': int(sum(len(v) for v in old))})
shift_function = save_table(pd.DataFrame(shift_rows), 'ctx_shift_function')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, rep in zip(axes, REPS):
    curve = shift_function[shift_function['representation'].eq(rep)]
    ax.plot(curve['quantile'], curve['delta'], marker='o', markersize=4, color=REP_PALETTE[rep])
    ax.fill_between(curve['quantile'], curve['ci_lo'], curve['ci_hi'],
                    color=REP_PALETTE[rep], alpha=0.20, linewidth=0)
    ax.axhline(curve['delta'].mean(), color='k', linewidth=0.9, linestyle=':')
    ax.axhline(0, color='k', linewidth=0.8)
    ax.annotate(f"tilt {curve['tilt'].iloc[0]:+.4f} (p={curve['p_tilt'].iloc[0]:.3f})",
                xy=(0.03, 0.05), xycoords='axes fraction', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    ax.set_xlabel('quantile level')
    ax.set_title(REP_LABEL[rep], fontsize=10)
axes[0].set_ylabel(f'centrality quantile, age {AGE_BANDS[-1]} minus age {AGE_BANDS[0]}')
plt.tight_layout()
save_fig('ctx_shift_function')
plt.show()

display(shift_function.groupby('representation')[['tilt', 'p_tilt', 'n_young', 'n_old']]
        .first().reindex(REPS).reset_index())

### 7.4 Distributional tests, and the shape of the within-story spread

Kolmogorov-Smirnov and Anderson-Darling ask whether the two banded distributions differ at all.
Brown-Forsythe, applied to centrality after each story's own mean has been subtracted, asks
specifically about spread. Because details are clustered in stories, the p-values these tests
return on pooled details are far too small, so each is repeated as a permutation test that
reassigns the band label at the story level, which is the only exchangeable unit available.

The story is then made the unit of analysis: within-story SD, interquartile range, skew and excess
kurtosis of centrality are regressed on centred source age with standard errors clustered by video,
with and without a length control. Slopes are plotted in units of each outcome's own between-story
SD so that five quantities on different scales can share an axis.

In [ ]:
rng = np.random.default_rng(SEED + 2)
test_rows = []
for rep in REPS:
    subset = banded[banded['representation'].eq(rep)].copy()
    subset['centred'] = subset['centrality'] - subset.groupby('story_key')['centrality'].transform('mean')
    pool = subset[subset['age_band'].isin([AGE_BANDS[0], AGE_BANDS[-1]])]
    young = pool[pool['age_band'].eq(AGE_BANDS[0])]
    old = pool[pool['age_band'].eq(AGE_BANDS[-1])]
    ks = stats.ks_2samp(young['centrality'], old['centrality'])
    anderson = stats.anderson_ksamp([young['centrality'].to_numpy(), old['centrality'].to_numpy()])
    brown = stats.levene(young['centred'], old['centred'], center='median')
    keys = pool[['story_key', 'age_band']].drop_duplicates('story_key')
    labels = keys['age_band'].eq(AGE_BANDS[-1]).to_numpy()
    by_story = {key: frame for key, frame in pool.groupby('story_key', sort=False)}
    raw_values = [by_story[key]['centrality'].to_numpy() for key in keys['story_key']]
    centred_values = [by_story[key]['centred'].to_numpy() for key in keys['story_key']]
    ks_null = np.empty(PERM_DRAWS)
    brown_null = np.empty(PERM_DRAWS)
    for draw in range(PERM_DRAWS):
        permuted = rng.permutation(labels)
        ks_null[draw] = stats.ks_2samp(
            np.concatenate([v for v, flag in zip(raw_values, permuted) if not flag]),
            np.concatenate([v for v, flag in zip(raw_values, permuted) if flag])).statistic
        brown_null[draw] = stats.levene(
            np.concatenate([v for v, flag in zip(centred_values, permuted) if not flag]),
            np.concatenate([v for v, flag in zip(centred_values, permuted) if flag]),
            center='median').statistic
    test_rows.append({'representation': rep, 'n_young': len(young), 'n_old': len(old),
                      'ks_stat': ks.statistic, 'ks_p_detail': ks.pvalue,
                      'ks_p_story_perm': float((ks_null >= ks.statistic).mean()),
                      'anderson_stat': anderson.statistic, 'anderson_p': anderson.pvalue,
                      'brown_forsythe_stat': brown.statistic, 'bf_p_detail': brown.pvalue,
                      'bf_p_story_perm': float((brown_null >= brown.statistic).mean()),
                      'within_sd_young': young['centred'].std(), 'within_sd_old': old['centred'].std(),
                      'within_sd_ratio_old_over_young': old['centred'].std() / young['centred'].std()})
distribution_tests = save_table(pd.DataFrame(test_rows), 'ctx_centrality_distribution_tests')

SHAPE_OUTCOMES = ['within_mean', 'within_sd', 'within_iqr', 'within_skew', 'within_kurtosis']
SHAPE_LABEL = ['mean', 'SD', 'IQR', 'skew', 'excess kurtosis']
shape_rows = []
for rep in REPS:
    subset = banded[banded['representation'].eq(rep)]
    moments = subset.groupby('story_key')['centrality'].agg(
        within_mean='mean', within_sd='std',
        within_iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
        within_skew=lambda s: stats.skew(s, bias=False),
        within_kurtosis=lambda s: stats.kurtosis(s, bias=False))
    joined = story_frame.merge(moments, on='story_key')
    for outcome in SHAPE_OUTCOMES:
        usable = joined.dropna(subset=[outcome, 'age_c', 'log_n'])
        for spec, formula in [('raw', f'{outcome} ~ age_c'),
                              ('length-controlled', f'{outcome} ~ age_c + log_n')]:
            model = smf.ols(formula, data=usable).fit(cov_type='cluster',
                                                     cov_kwds={'groups': usable['cluster']})
            shape_rows.append({'representation': rep, 'outcome': outcome, 'spec': spec, 'n': len(usable),
                               'level': usable[outcome].mean(), 'dispersion': usable[outcome].std(),
                               'slope': model.params['age_c'],
                               'ci_lo': model.conf_int().loc['age_c', 0],
                               'ci_hi': model.conf_int().loc['age_c', 1],
                               'p': model.pvalues['age_c']})
shape_slopes = save_table(pd.DataFrame(shape_rows), 'ctx_centrality_shape_slopes')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, rep in zip(axes, REPS):
    view = shape_slopes[shape_slopes['representation'].eq(rep)]
    positions = np.arange(len(SHAPE_OUTCOMES))
    for spec, offset, marker, colour in [('raw', -0.16, 'o', REP_PALETTE[rep]),
                                         ('length-controlled', 0.16, 's', MATCHED_COLOUR)]:
        block = view[view['spec'].eq(spec)].set_index('outcome').loc[SHAPE_OUTCOMES]
        scale = block['dispersion']
        ax.errorbar(block['slope'] / scale, positions + offset,
                    xerr=[(block['slope'] - block['ci_lo']) / scale,
                          (block['ci_hi'] - block['slope']) / scale],
                    fmt=marker, capsize=3, color=colour, label=spec, markersize=5)
    ax.axvline(0, color='k', linewidth=1, linestyle='--')
    ax.set_yticks(positions)
    ax.set_yticklabels(SHAPE_LABEL, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('age slope in between-story SD of the outcome (per year)')
    ax.set_title(REP_LABEL[rep], fontsize=10)
    ax.legend(fontsize=8)
plt.tight_layout()
save_fig('ctx_centrality_shape_slopes')
plt.show()

display(distribution_tests)
display(shape_slopes[shape_slopes['outcome'].isin(['within_sd', 'within_skew', 'within_kurtosis'])]
        [['representation', 'outcome', 'spec', 'level', 'slope', 'ci_lo', 'ci_hi', 'p']])

### 7.5 Direct measures of representational heterogeneity

Mean cosine cannot adjudicate the heterogeneity claim, because a lower mean cosine is the
observation being explained. These measures are computed from the normalised vectors instead, all of
them functions of a story's Gram matrix:

- **Participation ratio**, the square of the sum of eigenvalues over the sum of their squares, and
  the **entropy effective rank**, the exponential of the entropy of the normalised eigenvalues. Both
  count how many directions a story's details actually occupy. Centring matters here: without it the
  leading eigenvalue is the story's mean direction, and the measure mostly reports the anisotropy of
  the encoder rather than the spread of the story, so the eigenvalues are taken from the
  double-centred Gram matrix.
- **Mean and SD of the distance to the renormalised centroid**, one minus the cosine between a
  detail and the unit-length story mean.
- **SD of the off-diagonal pairwise cosine distribution.** Its mean is by definition the story's
  mean centrality, so only the SD carries new information.

One caveat has to be stated before the numbers are read. **Effective rank is bounded above by the
number of details**: a story with thirty details cannot occupy more than thirty directions, so a
longer story scores higher for purely mechanical reasons. The correlations with `log(n details)`
printed below make that concrete. Any age effect in the unmatched column is therefore
uninterpretable on its own, which is exactly why 7.6 is not optional.

In [ ]:
HETEROGENEITY = ['mean_cosine', 'sd_pairwise_cosine', 'participation_ratio', 'effective_rank',
                 'mean_dist_centroid', 'sd_dist_centroid']
HET_LABEL = {'mean_cosine': 'mean pairwise cosine', 'sd_pairwise_cosine': 'SD of pairwise cosine',
             'participation_ratio': 'participation ratio', 'effective_rank': 'entropy effective rank',
             'mean_dist_centroid': 'mean distance to centroid',
             'sd_dist_centroid': 'SD of distance to centroid'}

def heterogeneity(gram):
    size = gram.shape[0]
    upper = gram[np.triu_indices(size, 1)]
    double_centred = (gram - gram.mean(axis=0, keepdims=True)
                      - gram.mean(axis=1, keepdims=True) + gram.mean())
    eigenvalues = np.clip(np.linalg.eigvalsh(double_centred), 0.0, None)
    total = eigenvalues.sum()
    weights = eigenvalues[eigenvalues > 1e-12 * max(total, 1e-12)] / total
    distance = 1.0 - gram.mean(axis=1) / np.sqrt(max(gram.mean(), 1e-12))
    return {'mean_cosine': float(upper.mean()),
            'sd_pairwise_cosine': float(upper.std(ddof=1)),
            'participation_ratio': float(total ** 2 / np.square(eigenvalues).sum()),
            'effective_rank': float(np.exp(-(weights * np.log(weights)).sum())),
            'mean_dist_centroid': float(distance.mean()),
            'sd_dist_centroid': float(distance.std(ddof=1))}

def age_slopes(frame, estimator):
    collected = []
    for rep in REPS:
        subset = frame[frame['representation'].eq(rep)]
        for measure in HETEROGENEITY:
            usable = subset.dropna(subset=[measure, 'age_c', 'log_n'])
            for spec, formula in [('raw', f'{measure} ~ age_c'),
                                  ('length-controlled', f'{measure} ~ age_c + log_n')]:
                model = smf.ols(formula, data=usable).fit(cov_type='cluster',
                                                         cov_kwds={'groups': usable['cluster']})
                collected.append({'estimator': estimator, 'representation': rep, 'measure': measure,
                                  'spec': spec, 'n': len(usable), 'level': usable[measure].mean(),
                                  'slope': model.params['age_c'],
                                  'ci_lo': model.conf_int().loc['age_c', 0],
                                  'ci_hi': model.conf_int().loc['age_c', 1],
                                  'p': model.pvalues['age_c']})
    return pd.DataFrame(collected)

def slope_row(frame, estimator, rep, measure, spec):
    return frame[frame['estimator'].eq(estimator) & frame['representation'].eq(rep)
                 & frame['measure'].eq(measure) & frame['spec'].eq(spec)].iloc[0]

full_heterogeneity = pd.DataFrame(
    [{'story_key': key, 'representation': rep, **heterogeneity(gram_by_story[key][rep])}
     for key in story_frame['story_key'] for rep in REPS]).merge(story_frame, on='story_key')

length_dependence = save_table(
    pd.DataFrame([{'representation': rep, 'measure': measure,
                   'mean': full_heterogeneity.loc[full_heterogeneity['representation'].eq(rep), measure].mean(),
                   'pearson_with_log_n': full_heterogeneity[full_heterogeneity['representation'].eq(rep)]
                   [[measure, 'log_n']].corr().iloc[0, 1]}
                  for rep in REPS for measure in HETEROGENEITY]),
    'ctx_heterogeneity_length_dependence')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for column, rep in enumerate(REPS):
    subset = full_heterogeneity[full_heterogeneity['representation'].eq(rep)]
    for row, measure in enumerate(['participation_ratio', 'mean_dist_centroid']):
        ax = axes[row, column]
        binned = subset.groupby('source_age')[measure].agg(['mean', 'sem', 'size'])
        binned = binned[binned['size'] >= 5]
        ax.errorbar(binned.index, binned['mean'], yerr=1.96 * binned['sem'],
                    marker='o', color=REP_PALETTE[rep])
        correlation = length_dependence[length_dependence['representation'].eq(rep)
                                        & length_dependence['measure'].eq(measure)]['pearson_with_log_n'].iloc[0]
        ax.annotate(f'r with log n details = {correlation:+.2f}', xy=(0.03, 0.05),
                    xycoords='axes fraction', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
        ax.set_xlabel('source age (years)')
        ax.set_ylabel(HET_LABEL[measure])
        if row == 0:
            ax.set_title(REP_LABEL[rep], fontsize=10)
plt.tight_layout()
save_fig('ctx_heterogeneity_by_age')
plt.show()

display(length_dependence)

### 7.6 The decisive test: heterogeneity at matched story length

Two independent ways of taking length out of the estimator. The first is subsampling. For every
story with at least thirty details, one hundred random subsets of exactly thirty details are drawn,
every measure is recomputed inside each subset, and the results are averaged over draws. The
resulting quantity cannot depend on the number of details, because that number is identical for
every story by construction. The age regressions are then re-run on it. If the age effect survives,
older stories really are more heterogeneous detail for detail; if it disappears, the effect was
length all along.

Three estimators are compared: all details, all details restricted to the stories that qualify for
matching so that the sample is held fixed, and the matched quantity itself. The percentage of the
raw slope that survives matching is reported per measure, and left blank where the raw slope was not
significant to begin with. For the two rank measures that percentage should be read with the
n-dependence of 7.5 in mind — the unmatched slope there is inflated by construction, so what matters
is whether the matched slope still differs from zero, not what fraction of an inflated number it
represents.

In [ ]:
eligible = story_frame[story_frame['n_details'] >= MATCH_SIZE]
rng = np.random.default_rng(SEED + 3)
matched_rows = []
for story_key, n_details in zip(eligible['story_key'], eligible['n_details']):
    draws = [rng.choice(n_details, MATCH_SIZE, replace=False) for _ in range(MATCH_DRAWS)]
    for rep in REPS:
        gram = gram_by_story[story_key][rep]
        totals = {measure: 0.0 for measure in HETEROGENEITY}
        for index in draws:
            for measure, value in heterogeneity(gram[np.ix_(index, index)]).items():
                totals[measure] += value
        matched_rows.append({'story_key': story_key, 'representation': rep,
                             **{measure: totals[measure] / MATCH_DRAWS for measure in HETEROGENEITY}})
matched_heterogeneity = pd.DataFrame(matched_rows).merge(story_frame, on='story_key')

RESTRICTED = f'all details, n >= {MATCH_SIZE}'
MATCHED = f'matched to exactly {MATCH_SIZE}'
heterogeneity_slopes = save_table(
    pd.concat([age_slopes(full_heterogeneity, 'all details'),
               age_slopes(full_heterogeneity[full_heterogeneity['story_key'].isin(eligible['story_key'])],
                          RESTRICTED),
               age_slopes(matched_heterogeneity, MATCHED)], ignore_index=True),
    'ctx_heterogeneity_age_slopes')

survival_rows = []
for rep in REPS:
    for measure in HETEROGENEITY:
        raw = slope_row(heterogeneity_slopes, 'all details', rep, measure, 'raw')
        adjusted = slope_row(heterogeneity_slopes, 'all details', rep, measure, 'length-controlled')
        matched = slope_row(heterogeneity_slopes, MATCHED, rep, measure, 'raw')
        strict = slope_row(heterogeneity_slopes, MATCHED, rep, measure, 'length-controlled')
        interpretable = raw['p'] < 0.05
        survival_rows.append({
            'representation': rep, 'measure': measure,
            'raw_slope': raw['slope'], 'raw_p': raw['p'],
            'length_controlled_slope': adjusted['slope'], 'length_controlled_p': adjusted['p'],
            'matched_slope': matched['slope'], 'matched_p': matched['p'],
            'pct_of_raw_surviving_matching': 100.0 * matched['slope'] / raw['slope'] if interpretable else np.nan,
            'matched_and_controlled_slope': strict['slope'], 'matched_and_controlled_p': strict['p'],
            'pct_of_raw_surviving_both': 100.0 * strict['slope'] / raw['slope'] if interpretable else np.nan})
length_survival = save_table(pd.DataFrame(survival_rows), 'ctx_length_matched_survival')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for column, rep in enumerate(REPS):
    for row, measure in enumerate(['mean_cosine', 'participation_ratio']):
        ax = axes[row, column]
        for frame, label, style, colour in [(full_heterogeneity, 'all details', '-', REP_PALETTE[rep]),
                                            (matched_heterogeneity, MATCHED, '--', MATCHED_COLOUR)]:
            subset = frame[frame['representation'].eq(rep)]
            binned = subset.groupby('source_age')[measure].agg(['mean', 'sem', 'size'])
            binned = binned[binned['size'] >= 5]
            ax.errorbar(binned.index, binned['mean'] - subset[measure].mean(), yerr=1.96 * binned['sem'],
                        marker='o', markersize=4, linestyle=style, color=colour, label=label)
        summary = length_survival[length_survival['representation'].eq(rep)
                                  & length_survival['measure'].eq(measure)].iloc[0]
        tail = ('' if not np.isfinite(summary['pct_of_raw_surviving_matching'])
                else f"\n{summary['pct_of_raw_surviving_matching']:.0f}% of the raw slope survives")
        ax.axhline(0, color='k', linewidth=0.8)
        ax.annotate(f"all details {summary['raw_slope']:+.4f}/yr (p={summary['raw_p']:.1e})\n"
                    f"matched {summary['matched_slope']:+.4f}/yr (p={summary['matched_p']:.1e}){tail}",
                    xy=(0.03, 0.05), xycoords='axes fraction', fontsize=8,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
        ax.set_xlabel('source age (years)')
        ax.set_ylabel(f'{HET_LABEL[measure]}\n(deviation from its own overall mean)')
        ax.legend(fontsize=8)
        if row == 0:
            ax.set_title(REP_LABEL[rep], fontsize=10)
plt.tight_layout()
save_fig('ctx_length_matched_heterogeneity')
plt.show()

display(length_survival)

### 7.7 Length stratification, and a direct look at the confound

The second route makes no assumption about how length enters the model. Stories are split into
quartiles of the number of original details, the age slope on story mean cosine is estimated inside
each quartile, and the four estimates are combined by inverse-variance weighting with Cochran's Q as
a heterogeneity check. A significant Q means the age slope itself differs by story length, which is
worth knowing regardless of the pooled value.

The scatter is the confound in one picture. If the age bands separate vertically at a fixed position
on the horizontal axis, age is doing something that length does not already explain.

In [ ]:
full_heterogeneity['n_bin'] = pd.qcut(full_heterogeneity['n_details'], 4)
stratified_rows = []
for rep in REPS:
    subset = full_heterogeneity[full_heterogeneity['representation'].eq(rep)]
    for bin_label, frame in subset.groupby('n_bin', observed=True):
        model = smf.ols('mean_cosine ~ age_c', data=frame).fit(
            cov_type='cluster', cov_kwds={'groups': frame['cluster']})
        stratified_rows.append({'representation': rep, 'n_bin': str(bin_label), 'n_stories': len(frame),
                                'median_n_details': frame['n_details'].median(),
                                'slope': model.params['age_c'], 'se': model.bse['age_c'],
                                'ci_lo': model.conf_int().loc['age_c', 0],
                                'ci_hi': model.conf_int().loc['age_c', 1], 'p': model.pvalues['age_c']})
stratified = save_table(pd.DataFrame(stratified_rows), 'ctx_length_stratified_slopes')

pooled_rows = []
for rep in REPS:
    block = stratified[stratified['representation'].eq(rep)]
    weights = 1.0 / block['se'].to_numpy() ** 2
    pooled = float((weights * block['slope'].to_numpy()).sum() / weights.sum())
    standard_error = float(np.sqrt(1.0 / weights.sum()))
    cochran = float((weights * (block['slope'].to_numpy() - pooled) ** 2).sum())
    raw = slope_row(heterogeneity_slopes, 'all details', rep, 'mean_cosine', 'raw')
    pooled_rows.append({'representation': rep, 'n_bins': len(block), 'pooled_slope': pooled,
                        'se': standard_error, 'ci_lo': pooled - 1.96 * standard_error,
                        'ci_hi': pooled + 1.96 * standard_error, 'z': pooled / standard_error,
                        'p': float(2 * stats.norm.sf(abs(pooled / standard_error))),
                        'pct_of_raw_slope': 100.0 * pooled / raw['slope'],
                        'cochran_q': cochran, 'q_df': len(block) - 1,
                        'q_p': float(stats.chi2.sf(cochran, len(block) - 1))})
stratified_pooled = save_table(pd.DataFrame(pooled_rows), 'ctx_length_stratified_pooled')

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, rep in zip(axes, REPS):
    subset = full_heterogeneity[full_heterogeneity['representation'].eq(rep)]
    grid = np.linspace(subset['log_n'].min(), subset['log_n'].max(), 20)
    for band in AGE_BANDS:
        block = subset[subset['age_band'].eq(band)]
        ax.scatter(block['log_n'], block['mean_cosine'], s=14, alpha=0.6,
                   color=BAND_PALETTE[band], label=f'age {band}')
        ax.plot(grid, np.polyval(np.polyfit(block['log_n'], block['mean_cosine'], 1), grid),
                color=BAND_PALETTE[band], linewidth=1.8)
    ax.set_xlabel('log number of original details')
    ax.set_ylabel('story mean pairwise cosine')
    ax.set_title(REP_LABEL[rep], fontsize=10)
    ax.legend(fontsize=7)
plt.tight_layout()
save_fig('ctx_length_confound')
plt.show()

ESTIMATORS = [('all details', 'raw', 'age slope, all details'),
              ('all details', 'length-controlled', 'plus a log n control'),
              (RESTRICTED, 'raw', f'all details, stories with n >= {MATCH_SIZE}'),
              (MATCHED, 'raw', f'subsampled to exactly {MATCH_SIZE} details'),
              (MATCHED, 'length-controlled', f'subsampled to {MATCH_SIZE}, plus log n')]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for offset, rep in zip([-0.22, 0.0, 0.22], REPS):
    values, lows, highs = [], [], []
    for estimator, spec, _ in ESTIMATORS:
        row = slope_row(heterogeneity_slopes, estimator, rep, 'mean_cosine', spec)
        values.append(row['slope'])
        lows.append(row['slope'] - row['ci_lo'])
        highs.append(row['ci_hi'] - row['slope'])
    axes[0].errorbar(values, np.arange(len(ESTIMATORS)) + offset, xerr=[lows, highs], fmt='o',
                     capsize=3, color=REP_PALETTE[rep], label=REP_LABEL[rep], markersize=6)
    pooled = stratified_pooled[stratified_pooled['representation'].eq(rep)].iloc[0]
    axes[0].errorbar([pooled['pooled_slope']], [len(ESTIMATORS) + offset],
                     xerr=[[pooled['pooled_slope'] - pooled['ci_lo']],
                           [pooled['ci_hi'] - pooled['pooled_slope']]],
                     fmt='D', capsize=3, color=REP_PALETTE[rep], markersize=6)
axes[0].axvline(0, color='k', linestyle='--', linewidth=1)
axes[0].set_yticks(np.arange(len(ESTIMATORS) + 1))
axes[0].set_yticklabels([label for _, _, label in ESTIMATORS] + ['pooled within length quartile'], fontsize=8)
axes[0].invert_yaxis()
axes[0].set_xlabel('slope of story mean pairwise cosine on source age (per year)')
axes[0].set_title('Does the age effect survive length matching?', fontsize=10)
axes[0].legend(fontsize=7)

for rep in REPS:
    block = stratified[stratified['representation'].eq(rep)]
    axes[1].errorbar(block['median_n_details'], block['slope'],
                     yerr=[block['slope'] - block['ci_lo'], block['ci_hi'] - block['slope']],
                     marker='o', capsize=3, color=REP_PALETTE[rep], label=REP_LABEL[rep])
axes[1].axhline(0, color='k', linestyle='--', linewidth=1)
axes[1].set_xlabel('median number of original details in the length quartile')
axes[1].set_ylabel('age slope estimated inside the quartile')
axes[1].set_title('The age slope is not constant across story lengths', fontsize=10)
axes[1].legend(fontsize=7)
plt.tight_layout()
save_fig('ctx_length_matched_slopes')
plt.show()

display(stratified)
display(stratified_pooled)

### 7.8 How to read the four claims together

The four claims of 7 come apart cleanly, and the verdict cell states the outcome of each from the
computed values rather than from prose written in advance. The logic to apply to those numbers is
this. A constant-slope Wald test that fails to reject, together with a flat shift function, licenses
the word *entire*; a rejection means the distribution is changing shape and the word is wrong for
that space. A rank measure that still rises with age after the details have been subsampled to a
fixed count is evidence of genuine per-detail heterogeneity, because the estimator cannot see story
length at all. A slope that collapses under matching says the opposite, and the honest conclusion
in that case is that the age trend was a length trend wearing a developmental label.

Two further cautions. Within-story spread of centrality and heterogeneity of the underlying vectors
are not the same quantity and can move in opposite directions; the distribution of centralities can
narrow while the details themselves occupy more directions. And a significant Cochran's Q in 7.7
means the pooled within-quartile slope is an average over genuinely different slopes, so it should be
read alongside the per-quartile panel rather than instead of it.

## 8. A worked example

Numbers are easier to trust after seeing what a high-shift and a low-shift detail actually look
like. The story chosen below has enough details to be interesting and a clear spread in shift.

In [ ]:
candidates = (meta[meta['record_type'].eq('original')]
              .groupby('story_key')
              .agg(n=('detail_index', 'size'),
                   spread=('contextualization_shift', 'std'),
                   variety=('detail_text', lambda s: s.nunique() / len(s)))
              .query('n >= 25 and n <= 60 and variety >= 0.95')
              .sort_values('spread', ascending=False))
example_key = candidates.index[0]

example = details[details['story_key'].eq(example_key) & details['representation'].eq('contextual')]
example = example.merge(
    meta[meta['record_type'].eq('original')][['story_key', 'detail_index', 'detail_text']],
    left_on=['story_key', 'orig_idx'], right_on=['story_key', 'detail_index'])

print(f'story: {example_key}   details: {len(example)}   '
      f'retained: {int(example["retained"].sum())}')
print()
for label, frame in [('LOWEST contextualization shift (means the same alone as in context)',
                      example[example['orig_idx'] > 0].nsmallest(4, 'contextualization_shift')),
                     ('HIGHEST contextualization shift (only means something in context)',
                      example.nlargest(4, 'contextualization_shift'))]:
    print(label)
    print('-' * 100)
    for _, row in frame.iterrows():
        mark = 'KEPT   ' if row['retained'] else 'dropped'
        text = row['detail_text'][:88].replace('\n', ' ')
        print(f'  [{mark}] shift={row["contextualization_shift"]:.3f}  '
              f'centrality_pct={row["centrality_pct"]:.2f}  "{text}"')
    print()

## 9. Verdict

In [ ]:
gap_view = gap_table.set_index('representation')
full_model = survival_table[survival_table['spec'].eq('+ centrality')].set_index('term')
robust_model = survival_table[survival_table['spec'].eq('excl. first detail')].set_index('term')
kappa_min = agreement.groupby('representation')['cohens_kappa'].mean().min()
kappa_max = agreement.groupby('representation')['cohens_kappa'].mean().max()
rho_min = centrality_agreement['median_rho'].min()
rho_max = centrality_agreement['median_rho'].max()

lines = []
lines.append('1. THE COMPRESSION CONCLUSION IS NOT A TF-IDF ARTEFACT.')
lines.append('   Retained details sit higher in the within-story centrality ranking than omitted')
lines.append('   details in all three spaces:')
for rep in REPS:
    row = gap_view.loc[rep]
    lines.append(f'     {REP_LABEL[rep]:30s} gap={row["mean_gap"]:+.4f}  '
                 f'd_z={row["cohens_dz"]:+.2f}  p_holm={row["p_holm"]:.2e}')
lines.append('')

lines.append('2. BUT THE SPACES DISAGREE ABOUT WHICH SPECIFIC DETAILS THOSE ARE.')
lines.append(f'   Within-story centrality rank correlation between spaces: median rho '
             f'{rho_min:.2f} to {rho_max:.2f}.')
lines.append(f"   Agreement on the retained set: Cohen's kappa {kappa_min:.2f} to {kappa_max:.2f}.")
lines.append('   The aggregate claim is robust; any claim about an individual detail is not.')
lines.append('')

shift_or = full_model.loc['shift_z']
shift_or_robust = robust_model.loc['shift_z']
cent_or = full_model.loc['centrality_z']
lines.append('3. CONTEXT-DEPENDENCE IS A SECOND, INDEPENDENT SELECTION PRESSURE.')
lines.append(f'   Per +1 SD of contextualization shift, odds of being retained '
             f'{shift_or["odds_ratio"]:.3f} '
             f'[{shift_or["ci_lo"]:.3f}, {shift_or["ci_hi"]:.3f}], p={shift_or["p"]:.1e},')
lines.append('   holding centrality, position and length fixed')
lines.append(f'   (excluding first-in-story details: {shift_or_robust["odds_ratio"]:.3f}, '
             f'p={shift_or_robust["p"]:.1e}).')
lines.append(f'   Centrality moves the other way at {cent_or["odds_ratio"]:.3f} per +1 SD, and the two')
lines.append('   variables are close to uncorrelated. Summaries keep details that are typical of')
lines.append('   the story AND that stand on their own; they drop details whose meaning is')
lines.append('   manufactured by their surroundings.')
lines.append('')

lines.append('4. AGE EFFECTS ARE A STORY-LENGTH EFFECT, AGAIN.')
for outcome in ['mean_shift', 'retention', 'centrality_gap']:
    raw = age_table[(age_table['outcome'] == outcome) & (age_table['spec'] == 'raw')].iloc[0]
    adj = age_table[(age_table['outcome'] == outcome) & (age_table['spec'] == 'length-controlled')].iloc[0]
    verdict = 'survives' if adj['p'] < 0.05 else 'does not survive'
    lines.append(f'   {outcome:16s} raw {raw["slope_per_year"]:+.4f}/yr (p={raw["p"]:.1e})  ->  '
                 f'adj {adj["slope_per_year"]:+.4f}/yr (p={adj["p"]:.1e})  [{verdict}]')
lines.append('   Same pattern the age-steering notebook found: books for older readers are longer,')
lines.append('   and length rather than age drives the compression behaviour. Section 7 shows that this')
lines.append('   does not generalise to the centrality level, which is a separate outcome.')
lines.append('')

flat_raw = [rep for rep in REPS
            if quantile_constancy[quantile_constancy['representation'].eq(rep)
                                  & quantile_constancy['spec'].eq('raw')]['p_wald'].iloc[0] >= 0.05]
flat_adjusted = [rep for rep in REPS
                 if quantile_constancy[quantile_constancy['representation'].eq(rep)
                                       & quantile_constancy['spec'].eq('length-controlled')]['p_wald'].iloc[0] >= 0.05]
if len(flat_raw) == 0:
    lines.append('5. THE DOWNWARD AGE SHIFT IS NOT A PURE TRANSLATION IN ANY OF THE THREE SPACES.')
elif len(flat_raw) == 1:
    lines.append('5. THE DOWNWARD AGE SHIFT IS A PURE TRANSLATION IN ONLY ONE OF THE THREE SPACES.')
elif len(flat_raw) == len(REPS):
    lines.append('5. THE DOWNWARD AGE SHIFT IS A PURE TRANSLATION IN ALL THREE SPACES.')
else:
    lines.append(f'5. THE DOWNWARD AGE SHIFT IS A PURE TRANSLATION IN {len(flat_raw)} OF THE THREE SPACES.')
lines.append('   Decile slopes of detail centrality on source age, and a Wald test that they are all equal:')
for rep in REPS:
    row = quantile_constancy[quantile_constancy['representation'].eq(rep)
                             & quantile_constancy['spec'].eq('raw')].iloc[0]
    adjusted = quantile_constancy[quantile_constancy['representation'].eq(rep)
                                  & quantile_constancy['spec'].eq('length-controlled')].iloc[0]
    shape = 'pure location shift' if row['p_wald'] >= 0.05 else 'shape change'
    lines.append(f'     {REP_LABEL[rep]:30s} q10 {row["slope_q10"]:+.4f}  q90 {row["slope_q90"]:+.4f}  '
                 f'tilt {row["tilt_q90_minus_q10"]:+.4f} (p={row["p_tilt"]:.3f})  '
                 f'Wald p={row["p_wald"]:.1e}  [{shape}]')
    lines.append(f'     {"":30s} with a log n control, Wald p={adjusted["p_wald"]:.2f} '
                 f'and the decile spread falls from {row["slope_spread"]:.4f} to '
                 f'{adjusted["slope_spread"]:.4f}')
shift_tilts = shift_function.groupby('representation')[['tilt', 'p_tilt']].first()
lines.append(f'   The Doksum shift function between age {AGE_BANDS[0]} and age {AGE_BANDS[-1]} agrees: '
             f'tilt ' + ', '.join(f'{rep} {shift_tilts.loc[rep, "tilt"]:+.4f} '
                                  f'(p={shift_tilts.loc[rep, "p_tilt"]:.3f})' for rep in REPS) + '.')
lines.append(f'   Constant slopes across deciles hold unadjusted in {len(flat_raw)} of {len(REPS)} spaces '
             f'({", ".join(flat_raw) if flat_raw else "none"}) but in {len(flat_adjusted)} of {len(REPS)} '
             f'once length is controlled ({", ".join(flat_adjusted) if flat_adjusted else "none"}).')
if len(flat_adjusted) > len(flat_raw):
    lines.append('   So the fanning in the remaining spaces is itself a story-length artefact: at matched')
    lines.append('   length the whole distribution does translate, which is the one place the word')
    lines.append('   "entire" in the original claim is earned.')
else:
    lines.append('   Length control does not rescue the constant-slope claim, so the distribution is')
    lines.append('   changing shape and not merely sliding down.')
lines.append('')

lines.append('6. THE DISTRIBUTION NARROWS AS IT FALLS; IT DOES NOT SPREAD OUT.')
for rep in REPS:
    row = distribution_tests[distribution_tests['representation'].eq(rep)].iloc[0]
    sd_raw = shape_slopes[shape_slopes['representation'].eq(rep) & shape_slopes['outcome'].eq('within_sd')
                          & shape_slopes['spec'].eq('raw')].iloc[0]
    sd_adj = shape_slopes[shape_slopes['representation'].eq(rep) & shape_slopes['outcome'].eq('within_sd')
                          & shape_slopes['spec'].eq('length-controlled')].iloc[0]
    lines.append(f'     {REP_LABEL[rep]:30s} KS D={row["ks_stat"]:.3f} (story-permutation '
                 f'p={row["ks_p_story_perm"]:.3f}), Anderson-Darling A2={row["anderson_stat"]:.1f}')
    lines.append(f'     {"":30s} within-story SD ratio old/young {row["within_sd_ratio_old_over_young"]:.2f}, '
                 f'Brown-Forsythe p={row["bf_p_story_perm"]:.3f}')
    lines.append(f'     {"":30s} within-story SD slope {sd_raw["slope"]:+.4f}/yr (p={sd_raw["p"]:.1e})  ->  '
                 f'length-controlled {sd_adj["slope"]:+.4f}/yr (p={sd_adj["p"]:.1e})')
shape_moved = shape_slopes[shape_slopes['outcome'].isin(['within_skew', 'within_kurtosis'])
                           & shape_slopes['spec'].eq('raw') & (shape_slopes['p'] < 0.05)]
lines.append(f'   Skew and excess kurtosis move with age in {len(shape_moved)} of '
             f'{2 * len(REPS)} representation-by-moment tests, so the tails are not reshaping.')
lines.append('   A lower and narrower centrality distribution is not the same thing as a more spread')
lines.append('   out one, and the hypothesis as worded implies the latter.')
lines.append('')

lines.append('7. OLDER STORIES ARE MORE HETEROGENEOUS PER DETAIL, BUT LENGTH DOES MOST OF THE WORK.')
lines.append(f'   Length matching draws {MATCH_DRAWS} subsets of exactly {MATCH_SIZE} details per story, '
             f'for the {len(eligible)} of {len(story_frame)} stories that qualify, so the estimator cannot')
lines.append('   see story length at all.')
for rep in REPS:
    for measure in ['mean_cosine', 'participation_ratio', 'mean_dist_centroid']:
        row = length_survival[length_survival['representation'].eq(rep)
                              & length_survival['measure'].eq(measure)].iloc[0]
        share = ('n/a' if not np.isfinite(row['pct_of_raw_surviving_matching'])
                 else f'{row["pct_of_raw_surviving_matching"]:.0f}% of raw')
        lines.append(f'     {rep:11s} {HET_LABEL[measure]:26s} raw {row["raw_slope"]:+.4f}/yr '
                     f'(p={row["raw_p"]:.1e})  ->  matched {row["matched_slope"]:+.4f}/yr '
                     f'(p={row["matched_p"]:.1e}), {share}')
for measure in ['participation_ratio', 'effective_rank', 'mean_dist_centroid']:
    survivors = length_survival[length_survival['measure'].eq(measure)
                                & (length_survival['matched_p'] < 0.05)
                                & (length_survival['matched_slope'] > 0)]['representation'].tolist()
    lines.append(f'   {HET_LABEL[measure]} still rises with age after matching in {len(survivors)} of '
                 f'{len(REPS)} spaces ({", ".join(survivors) if survivors else "none"}).')
spread_survivors = length_survival[length_survival['measure'].eq('sd_pairwise_cosine')
                                   & (length_survival['matched_p'] < 0.05)
                                   & (length_survival['matched_slope'] > 0)]['representation'].tolist()
lines.append(f'   The SD of the pairwise cosine distribution rises with age in '
             f'{len(spread_survivors)} of {len(REPS)} spaces, so the details occupy more directions')
lines.append('   without the spread of their similarities widening. The heterogeneity claim is')
lines.append('   therefore not empty, but the surviving slopes are a fraction of the unmatched ones,')
lines.append('   and the raw rank slopes are inflated by the fact that effective rank is bounded above')
lines.append('   by the number of details in the first place.')
for rep in REPS:
    row = stratified_pooled[stratified_pooled['representation'].eq(rep)].iloc[0]
    lines.append(f'     {rep:11s} pooled within length quartile {row["pooled_slope"]:+.4f}/yr '
                 f'(p={row["p"]:.1e}), {row["pct_of_raw_slope"]:.0f}% of the raw slope, '
                 f'Cochran Q p={row["q_p"]:.3f}')
lines.append('')

lines.append('8. WHAT THIS MEANS FOR THE AGE TRENDS IN SECTION 6.')
retention_raw = age_table[(age_table['outcome'] == 'retention') & (age_table['spec'] == 'raw')].iloc[0]
retention_adj = age_table[(age_table['outcome'] == 'retention')
                          & (age_table['spec'] == 'length-controlled')].iloc[0]
centrality_survival = length_survival[length_survival['representation'].eq('contextual')
                                      & length_survival['measure'].eq('mean_cosine')].iloc[0]
lines.append(f'   Retention rate lost most of its age slope to the length control '
             f'({retention_raw["slope_per_year"]:+.4f} -> {retention_adj["slope_per_year"]:+.4f}/yr, '
             f'p={retention_adj["p"]:.3f}), but centrality does not behave the same way:')
lines.append(f'   {centrality_survival["pct_of_raw_surviving_matching"]:.0f}% of the contextual centrality '
             f'slope survives fixed-size matching '
             f'({centrality_survival["matched_slope"]:+.4f}/yr, p={centrality_survival["matched_p"]:.1e}).')
dense_survival = length_survival[length_survival['measure'].eq('mean_cosine')
                                 & length_survival['representation'].isin(['isolated', 'contextual'])]
lines.append('   The age decline in centrality is therefore not reducible to length in the way the')
lines.append('   retention and shift results were. The correct reading of section 6 is that older')
lines.append('   stories place their details further apart and spread them over more directions, at')
lines.append(f'   {dense_survival["pct_of_raw_surviving_matching"].min():.0f} to '
             f'{dense_survival["pct_of_raw_surviving_matching"].max():.0f} per cent of the scale the')
lines.append('   unadjusted slope advertises in the two dense spaces, while the spread of centralities')
lines.append('   within a story narrows rather than widens.')
lines.append('')

lines.append('9. WHAT STILL NEEDS THE HPC RUN.')
lines.append('   MiniLM prefix conditioning is a proxy for GPT-OSS causal conditioning. It')
lines.append('   establishes that the contextual/isolated distinction changes the selected set and')
lines.append('   that context-dependence predicts omission. It cannot say how a 20B causal decoder')
lines.append('   distributes shift across layers, nor reproduce the GPT-OSS centrality values that')
lines.append('   gpt_oss_detail_centrality_retention.ipynb reports. Run:')
lines.append('     sbatch hpc/run_gpt_oss_detail_representation_alignment.slurm')
lines.append('     sbatch hpc/run_gpt_oss_detail_centrality.slurm')
lines.append('   then sync mechanistic_interpretability/outputs/02_representation_extraction/.')

report = '\n'.join(lines)
print(report)
out_path = TABLE_DIR / 'ctx_vs_isolated_verdict.txt'
out_path.write_text(report, encoding='utf-8')
print()
print(f'saved -> {out_path}')